# Section 3: The MTTR Race — Real Exploitation Timeline Analysis

**ITESO | Vulnerability Management and the AI Crossroads**

This notebook uses a technique that isn't often demonstrated in lectures: **combining the CISA KEV catalog with NVD publication dates to measure real exploitation lag times** — how long it takes from the day a CVE is publicly disclosed to the day CISA confirms it is being actively exploited.

**What we'll measure:**
- The distribution of exploitation lag across ~1,200 KEV entries
- Whether recent CVEs are being exploited faster (the AI acceleration effect)
- Which vulnerability types get exploited fastest
- The Window of Exposure at different patching SLAs

**Why this matters:** If exploitation lag is shrinking, your 30-day or 90-day patching SLA may no longer be fast enough to stay ahead of active exploitation.

In [ ]:
import requests
import json
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

OUTPUT = Path('output')
OUTPUT.mkdir(exist_ok=True)

KEV_URL = 'https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'
NVD_API = 'https://services.nvd.nist.gov/rest/json/cves/2.0'
NVD_API_KEY = None  # optional — set to speed up NVD queries

print('Setup complete.')

## 1. Load the KEV Catalog

We use the cached KEV catalog if available (from notebook 02).

In [ ]:
KEV_CACHE = OUTPUT / 'kev_catalog.json'

if KEV_CACHE.exists():
    with open(KEV_CACHE) as f:
        kev_raw = json.load(f)
    print('Loaded KEV from cache.')
else:
    print('Downloading KEV catalog...')
    r = requests.get(KEV_URL, timeout=30)
    r.raise_for_status()
    kev_raw = r.json()
    with open(KEV_CACHE, 'w') as f:
        json.dump(kev_raw, f)

kev = pd.DataFrame(kev_raw['vulnerabilities'])
kev['dateAdded'] = pd.to_datetime(kev['dateAdded'])
kev['year_added'] = kev['dateAdded'].dt.year
print(f'{len(kev):,} KEV entries loaded.')

## 2. Fetching NVD Publication Dates

For each KEV entry, we query the NVD API for the CVE's original publication date — the day the vulnerability was first publicly disclosed. The **exploitation lag** is then:

```
exploitation_lag = kev.dateAdded - nvd.published
```

This is a proxy for how long it takes a vulnerability to go from "publicly known" to "confirmed in-the-wild exploitation." We sample 300 entries for speed (the cache will cover all of them after first run).

In [ ]:
NVD_DATES_CACHE = OUTPUT / 'nvd_published_dates.json'

def fetch_nvd_published(cve_id: str, api_key=None) -> str | None:
    """Return the NVD published date for a CVE, or None on failure."""
    headers = {'apiKey': api_key} if api_key else {}
    try:
        r = requests.get(NVD_API, params={'cveId': cve_id}, headers=headers, timeout=15)
        r.raise_for_status()
        vulns = r.json().get('vulnerabilities', [])
        if vulns:
            return vulns[0]['cve']['published']  # ISO 8601 string
    except Exception:
        pass
    return None


if NVD_DATES_CACHE.exists():
    with open(NVD_DATES_CACHE) as f:
        nvd_dates = json.load(f)
    print(f'Loaded NVD date cache: {len(nvd_dates):,} entries')
else:
    # Sample across years so we get good temporal coverage
    sample = kev.groupby('year_added', group_keys=False).apply(
        lambda g: g.sample(min(len(g), 40), random_state=42)
    ).reset_index(drop=True)
    sample_ids = sample['cveID'].tolist()
    print(f'Fetching NVD dates for {len(sample_ids)} sampled CVEs...')
    print(f'(~{len(sample_ids) * (1 if NVD_API_KEY else 7)}s — set NVD_API_KEY to go faster)')
    nvd_dates = {}
    delay = 0.7 if NVD_API_KEY else 7
    for i, cve_id in enumerate(sample_ids):
        pub_date = fetch_nvd_published(cve_id, NVD_API_KEY)
        if pub_date:
            nvd_dates[cve_id] = pub_date
        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(sample_ids)} fetched...')
        time.sleep(delay)
    with open(NVD_DATES_CACHE, 'w') as f:
        json.dump(nvd_dates, f)
    print(f'Cached {len(nvd_dates)} dates to {NVD_DATES_CACHE}')

print(f'\nNVD publication dates available for: {len(nvd_dates):,} CVEs')

## 3. Computing Exploitation Lag

In [ ]:
kev['nvd_published'] = pd.to_datetime(
    kev['cveID'].map(nvd_dates), utc=True, errors='coerce'
).dt.tz_localize(None)

kev['dateAdded_naive'] = kev['dateAdded'].dt.tz_localize(None)
kev['exploit_lag_days'] = (kev['dateAdded_naive'] - kev['nvd_published']).dt.days

lag_df = kev.dropna(subset=['exploit_lag_days']).copy()
# Filter out negative lags (CVEs added to KEV before NVD published — data quality edge case)
lag_df = lag_df[lag_df['exploit_lag_days'] >= 0]

print(f'Exploitation lag analysis ({len(lag_df)} CVEs with valid data):\n')
desc = lag_df['exploit_lag_days'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
for k, v in desc.items():
    label = {'count': 'n', 'mean': 'Mean', 'std': 'Std dev',
             '10%': '10th pct', '25%': '25th pct', '50%': 'Median',
             '75%': '75th pct', '90%': '90th pct', 'min': 'Min', 'max': 'Max'}.get(k, k)
    print(f'  {label:<12}: {v:.0f} days')

fast_exploit = (lag_df['exploit_lag_days'] <= 7).mean() * 100
within_30    = (lag_df['exploit_lag_days'] <= 30).mean() * 100
within_90    = (lag_df['exploit_lag_days'] <= 90).mean() * 100

print(f'\nExploited within  7 days of disclosure: {fast_exploit:.1f}%')
print(f'Exploited within 30 days of disclosure: {within_30:.1f}%')
print(f'Exploited within 90 days of disclosure: {within_90:.1f}%')

## 4. Has the Exploitation Timeline Changed Over Time?

If AI-assisted exploit generation is accelerating, we'd expect to see the median exploitation lag decrease in more recent years.

In [ ]:
yearly_lag = (
    lag_df.groupby('year_added')['exploit_lag_days']
    .agg(['median', 'mean', 'count'])
    .reset_index()
)
# Only show years with enough samples
yearly_lag = yearly_lag[yearly_lag['count'] >= 5]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.subplots_adjust(top=0.88, bottom=0.14)

# Left: exploitation lag distribution (log scale)
ax = axes[0]
lag_clipped = lag_df['exploit_lag_days'].clip(upper=365 * 3)
ax.hist(lag_clipped, bins=60, color='#4a90d9', alpha=0.85, edgecolor='white', lw=0.3)
ax.axvline(7,  color='#e06c5a', lw=2, linestyle='--', label='7 days')
ax.axvline(30, color='#f0a500', lw=2, linestyle='--', label='30 days (typical critical SLA)')
ax.axvline(90, color='#60b26e', lw=2, linestyle='--', label='90 days (typical high SLA)')
med = lag_df['exploit_lag_days'].median()
ax.axvline(med, color='#2c2c2c', lw=1.5, linestyle='-.', label=f'Median = {med:.0f} days')
ax.set_xlabel('Days from NVD Publication to KEV Entry')
ax.set_ylabel('Number of CVEs')
ax.set_title('Exploitation Lag Distribution\n(NVD disclosure → CISA KEV confirmation)', fontweight='bold')
ax.legend(fontsize=8)
ax.set_xlim(0, None)
ax.grid(axis='y', alpha=0.3)

# Right: median exploitation lag by year
ax2 = axes[1]
ax2.bar(yearly_lag['year_added'], yearly_lag['median'],
        color='#4a90d9', alpha=0.85, label='Median lag (days)')
ax2.plot(yearly_lag['year_added'], yearly_lag['median'],
         'ko-', ms=5, lw=1.5)

# Trend line
if len(yearly_lag) >= 3:
    slope, intercept, *_ = stats.linregress(yearly_lag['year_added'], yearly_lag['median'])
    xs = np.array(yearly_lag['year_added'])
    ax2.plot(xs, slope * xs + intercept, 'r--', lw=1.5, alpha=0.7,
             label=f'Trend ({slope:+.0f} days/yr)')

ax2.set_xlabel('Year CVE Added to KEV')
ax2.set_ylabel('Median Exploitation Lag (days)')
ax2.set_title('Is the Exploitation Timeline Shrinking?\n(Median days to confirmed exploitation by year)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

for _, row in yearly_lag.iterrows():
    ax2.text(row['year_added'], row['median'] + 5, f'{row["median"]:.0f}',
             ha='center', va='bottom', fontsize=8)

fig.suptitle('Real Exploitation Timelines from CISA KEV + NVD Data',
             fontsize=13, fontweight='bold')
fig.text(0.5, 0.02,
         'Source: CISA KEV dateAdded vs. NVD published date. '
         'Lag = days from public disclosure to confirmed exploitation. '
         'Sample may vary from full population.',
         ha='center', fontsize=8, color='#666', style='italic',
         transform=fig.transFigure)
plt.savefig(OUTPUT / '03_exploitation_timeline.png', bbox_inches='tight')
plt.show()
print('Chart saved.')

## 5. SLA Adequacy Analysis

Given the real exploitation lag distribution, what percentage of actively-exploited CVEs would your patching SLA catch *before* exploitation was confirmed?

**This is the core question** for any vulnerability management program: is your SLA fast enough?

In [ ]:
# --- PARAMETER: your organisation's patching SLAs (days) ---
SLA_CRITICAL = 14   # days to patch critical CVEs
SLA_HIGH     = 30   # days to patch high CVEs
SLA_MEDIUM   = 90   # days to patch medium CVEs

# What % of KEV entries would each SLA have caught before confirmed exploitation?
total = len(lag_df)

print('SLA Adequacy Analysis')
print('=' * 60)
print(f'\nBased on {total} KEV entries with measured exploitation lag:')
print()

for label, sla_days in [('Critical SLA', SLA_CRITICAL), ('High SLA', SLA_HIGH),
                          ('Medium SLA', SLA_MEDIUM)]:
    caught = (lag_df['exploit_lag_days'] > sla_days).sum()
    missed = total - caught
    pct_caught = caught / total * 100
    print(f'{label} ({sla_days} days):')
    print(f'  Exploitation lag > SLA (patch wins):  {caught:>5,} ({pct_caught:.1f}%)')
    print(f'  Exploitation lag ≤ SLA (race lost):   {missed:>5,} ({100-pct_caught:.1f}%)')
    print()

print(f'\nKey takeaway:')
print(f'  {(lag_df["exploit_lag_days"] <= 7).sum()} CVEs ({fast_exploit:.1f}%) were exploited within 7 days.')
print(f'  No SLA can protect against these without near-zero-day patching capability.')
print(f'  Compensating controls (network segmentation, WAF, MFA) are required for fast-exploit CVEs.')

## 6. The MTTR Race: WoE Visualisation

The Window of Exposure (WoE) is the time between "exploit available" and "patch deployed." We can now overlay real exploitation timing against typical deployment timelines.

In [ ]:
# Percentile-based exploitation timeline from real data
p10 = float(lag_df['exploit_lag_days'].quantile(0.10))
p25 = float(lag_df['exploit_lag_days'].quantile(0.25))
p50 = float(lag_df['exploit_lag_days'].quantile(0.50))

# Illustrative deployment timelines (days from patch available to deployed)
# Patch availability lag from NVD publication: roughly 1-14 days for critical issues
scenarios = {
    'Fast org\n(cloud-native)':  {'patch_avail': 5,  'deploy': 14},
    'Typical org\n(mixed estate)': {'patch_avail': 14, 'deploy': 45},
    'Slow org\n(legacy/regulated)': {'patch_avail': 30, 'deploy': 120},
}

fig, ax = plt.subplots(figsize=(13, 5))
fig.subplots_adjust(top=0.85, bottom=0.18)

y_pos = list(range(len(scenarios)))
colors = ['#60b26e', '#f0a500', '#e06c5a']

# Exploitation timing markers
ax.axvline(p10, color='#888', lw=1.2, linestyle=':', alpha=0.8)
ax.axvline(p25, color='#888', lw=1.2, linestyle=':', alpha=0.8)
ax.axvline(p50, color='#888', lw=1.2, linestyle=':', alpha=0.8)
ax.text(p10 + 0.5, len(scenarios) - 0.15, f'10th pct\n{p10:.0f}d', fontsize=7.5, color='#555')
ax.text(p25 + 0.5, len(scenarios) - 0.15, f'25th pct\n{p25:.0f}d', fontsize=7.5, color='#555')
ax.text(p50 + 0.5, len(scenarios) - 0.15, f'Median\n{p50:.0f}d', fontsize=7.5, color='#555')

for i, (label, timing) in enumerate(scenarios.items()):
    patch_day  = timing['patch_avail']
    deploy_day = timing['deploy']
    # Window of Exposure = time between earliest exploitation (p10) and deployment
    woe_start = p10
    woe_end   = deploy_day
    
    # Patch timeline bar
    ax.barh(i, patch_day, left=0, height=0.5,
            color='#4a90d9', alpha=0.85, label='Patch available' if i == 0 else '')
    ax.barh(i, deploy_day - patch_day, left=patch_day, height=0.5,
            color='#aac8e8', alpha=0.85, label='Deployment lag' if i == 0 else '')
    
    # Window of Exposure
    if woe_end > woe_start:
        ax.barh(i - 0.3, woe_end - woe_start, left=woe_start, height=0.25,
                color='#e06c5a', alpha=0.5)
        ax.text((woe_start + woe_end) / 2, i - 0.3,
                f'WoE = {woe_end - woe_start:.0f}d', ha='center', va='center',
                fontsize=7.5, color='#c0392b', fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(list(scenarios.keys()), fontsize=10)
ax.set_xlabel('Days from CVE Public Disclosure')
ax.set_title('MTTR Race: Deployment Timeline vs. Real Exploitation Timing\n'
             '(Exploitation percentiles from CISA KEV + NVD data)', fontweight='bold')

woe_patch = mpatches.Patch(color='#e06c5a', alpha=0.5, label='Window of Exposure (WoE)')
patch_bar = mpatches.Patch(color='#4a90d9', alpha=0.85, label='Patch available')
deploy_bar = mpatches.Patch(color='#aac8e8', alpha=0.85, label='Deployment lag')
ax.legend(handles=[patch_bar, deploy_bar, woe_patch], loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

fig.text(0.5, 0.02,
         'Exploitation percentiles are real measurements from KEV+NVD. '
         'Patch timelines are illustrative. Third-party/vendor software follows vendor release cycles.',
         ha='center', fontsize=8, color='#666', style='italic', transform=fig.transFigure)
plt.savefig(OUTPUT / '03_mttr_race.png', bbox_inches='tight')
plt.show()
print('Chart saved.')

## 7. Try It Yourself

Change `SLA_CRITICAL`, `SLA_HIGH`, and `SLA_MEDIUM` in cell 5 to match your organisation's actual SLAs and re-run. How does your SLA compare to the real exploitation timeline?